In [1]:
import torch
import torch.nn as nn
from abc import ABC, abstractmethod
from transformers import AutoModel
from peft import LoraConfig, get_peft_model


In [2]:

class BaseForecaster(nn.Module, ABC):
    """
    Abstract base class for all time-series forecasting models in this project.
    Ensures that all models, from baselines to foundation models, share the same API.
    """
    def __init__(self, input_len: int, output_len: int):
        super().__init__()
        self.input_len = input_len
        self.output_len = output_len

    @abstractmethod
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, channels, input_len)
        Returns:
            Tensor of shape (batch_size, output_len)
        """
        pass
    """
Module Name: timesfm.py
Description: TimesFM models (Zero-shot, Linear Head Finetuning, LoRA Finetuning)
"""




In [24]:
from transformers import TimesFm2_5ModelForPrediction
# ============================================================
# Base Wrapper
# ============================================================
class TimesFMBase(BaseForecaster):
    def __init__(self, input_len: int, output_len: int, model_name: str):
        super().__init__(input_len, output_len)
        self.output_len = output_len
        
        # Load the model INCLUDING the pre-trained forecasting head
        self.model = TimesFm2_5ModelForPrediction.from_pretrained(model_name)

# ============================================================
# 1. ZERO-SHOT
# ============================================================
class TimesFMZeroShot(TimesFMBase):
    def __init__(self, input_len, output_len, model_name, random_seed=42):
        super().__init__(input_len, output_len, model_name)

        # Freeze everything
        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, x):
        """
        x: (B, 1, L)
        """
        x = x.squeeze(1)
        
        # TimesFM native forward pass (handles internal denormalization)
        outputs = self.model(past_values=x)
        
        # Extract the point forecasts (mean predictions)
        return outputs.mean_predictions[:, -self.output_len:]


# ============================================================
# 2. NATIVE HEAD FINETUNING (Replaces Linear Head)
# ============================================================
class TimesFMLinearHead(TimesFMBase):
    def __init__(self, input_len, output_len, model_name, random_seed=42):
        super().__init__(input_len, output_len, model_name)

        # Freeze the entire backbone first
        for p in self.model.parameters():
            p.requires_grad = False

        # Unfreeze ONLY the native forecasting head. 
        # This is strictly better than an nn.Linear layer because it 
        # preserves TimesFM's structural patching and de-normalization.
        for name, p in self.model.named_parameters():
            if any(keyword in name for keyword in ["head", "scoring", "output_layer"]):
                p.requires_grad = True

    def forward(self, x):
        x = x.squeeze(1)
        outputs = self.model(past_values=x)
        return outputs.mean_predictions[:, -self.output_len:]


# ============================================================
# 3. LORA FINETUNING
# ============================================================
class TimesFMLoRA(TimesFMBase):
    def __init__(self,
                 input_len,
                 output_len,
                 model_name,
                 r=8,
                 lora_alpha=16,
                 lora_dropout=0.05,
                 random_seed=42):

        super().__init__(input_len, output_len, model_name)

        # Freeze base first
        for p in self.model.parameters():
            p.requires_grad = False

        # LoRA config targeting typical Hugging Face attention projections
        lora_config = LoraConfig(
            r=r,
            lora_alpha=lora_alpha,
            target_modules=["q_proj", "v_proj"], 
            lora_dropout=lora_dropout,
            bias="none"
        )

        # Inject LoRA adapters into the backbone
        self.model = get_peft_model(self.model, lora_config)

        # Ensure the native forecasting head is also unfrozen so it can 
        # learn alongside the new LoRA adapters
        for name, p in self.model.named_parameters():
            if any(keyword in name for keyword in ["head", "scoring", "output_layer"]):
                p.requires_grad = True

        print("LoRA adapters added to TimesFM.")
        self.model.print_trainable_parameters()

    def forward(self, x):
        x = x.squeeze(1)
        outputs = self.model(past_values=x)
        return outputs.mean_predictions[:, -self.output_len:]

In [29]:
model_name = "google/timesfm-2.5-200m-transformers"
input_len = 32
output_len = 9
zero_shot_model = TimesFMLoRA(input_len, output_len, model_name)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

LoRA adapters added to TimesFM.
trainable params: 17,204,480 || all params: 232,108,480 || trainable%: 7.4123


In [30]:
x = torch.randn(5, 1, input_len)  # (B, C, L)
with torch.no_grad():
    out = zero_shot_model(x)
print(out.shape)
out

torch.Size([5, 9])


tensor([[-0.2142, -0.2039, -0.2215, -0.2624, -0.2624, -0.2777, -0.2626, -0.2619,
         -0.2561],
        [-0.1495, -0.1752, -0.1895, -0.1509, -0.1578, -0.1702, -0.1424, -0.1314,
         -0.1503],
        [-0.2133, -0.2017, -0.1986, -0.2173, -0.1932, -0.2006, -0.2052, -0.1747,
         -0.1874],
        [-0.0969, -0.0973, -0.0926, -0.0834, -0.1082, -0.0910, -0.0754, -0.0929,
         -0.1122],
        [ 0.2854,  0.2841,  0.2847,  0.2131,  0.2064,  0.1980,  0.2375,  0.1881,
          0.1640]])